In [2]:
! pip install compress_pickle

DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/music_sdk_python3-2.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/papermill-2.3.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from compress_pickle import load

DATA_PATH = "./2days"

flight_data_path = f"{DATA_PATH}/flight_data.pkl"
flight_header_path = f"{DATA_PATH}/flight_header.csv"
stats_path = f"{DATA_PATH}/stats.csv"

print("Loading data...")

flight_data = load(flight_data_path)
header_df = pd.read_csv(flight_header_path, index_col="Master Index")
stats_df = pd.read_csv(stats_path)

print("Loaded.")

num_flights = len(header_df)
channels = 23
max_length = 4096

print("Number of flights:", num_flights)

# ==============================
# flight length
# ==============================

lengths = []
nan_counts = []

total_values = 0
total_nan = 0

for idx in header_df.index:
    arr = flight_data[idx]   # shape: (T, 23)
    
    T, C = arr.shape
    lengths.append(T)

    nan_count = np.isnan(arr).sum()
    nan_counts.append(nan_count)

    total_values += arr.size
    total_nan += nan_count

lengths = np.array(lengths)
nan_counts = np.array(nan_counts)

print("\n===== Flight Length Statistics =====")
print("Min length:", lengths.min())
print("Max length:", lengths.max())
print("Mean length:", lengths.mean())
print("Median length:", np.median(lengths))
print("Std length:", lengths.std())

# ==============================
# NaN 
# ==============================

print("\n===== NaN Statistics =====")

print("Total values:", total_values)
print("Total NaN:", total_nan)

print("Overall NaN ratio:", total_nan / total_values)

print("Flights containing NaN:",
      (nan_counts > 0).sum(), "/", num_flights)

print("Ratio flights with NaN:",
      (nan_counts > 0).mean())

# ==============================
# padding 
# ==============================

padding_length = np.maximum(0, max_length - lengths)
padding_ratio = padding_length / max_length

print("\n===== Padding Statistics (last 4096 strategy) =====")

print("Flights shorter than 4096:",
      (lengths < max_length).sum())

print("Ratio shorter than 4096:",
      (lengths < max_length).mean())

print("Average padding ratio:",
      padding_ratio.mean())

print("Max padding length:",
      padding_length.max())


print("\n===== Label Distribution =====")

print(header_df["before_after"].value_counts())
print("\nRatio:")
print(header_df["before_after"].value_counts(normalize=True))


plt.figure(figsize=(8,5))

plt.hist(lengths, bins=50)
plt.axvline(max_length, linestyle="--")

plt.title("Flight Length Distribution")
plt.xlabel("Flight Length (timesteps)")
plt.ylabel("Number of Flights")

plt.show()



plt.figure(figsize=(8,5))

plt.hist(padding_ratio, bins=50)

plt.title("Padding Ratio Distribution")
plt.xlabel("Padding Ratio")
plt.ylabel("Number of Flights")

plt.show()

Loading data...
Loaded.
Number of flights: 11446

===== Flight Length Statistics =====
Min length: 605
Max length: 20679
Mean length: 5517.537742442775
Median length: 5282.5
Std length: 2211.067550453348

===== NaN Statistics =====
Total values: 1452535951
Total NaN: 15667510
Overall NaN ratio: 0.010786314782235637
Flights containing NaN: 8380 / 11446
Ratio flights with NaN: 0.7321334964179627

===== Padding Statistics (last 4096 strategy) =====
Flights shorter than 4096: 2521
Ratio shorter than 4096: 0.22025161628516512
Average padding ratio: 0.06016456008103267
Max padding length: 3491

===== Label Distribution =====
before_after
0    5844
1    5602
Name: count, dtype: int64

Ratio:
before_after
0    0.510571
1    0.489429
Name: proportion, dtype: float64


<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>